# Hub のモデルのキャッシュ場所を確認・変更する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/08_transformers/hf_cache.ipynb)

NLP クックブック 08-09 ／ Transformers の基本操作

ダウンロードしたモデルがどこに保存されているか知りたい。ディスクを圧迫しているキャッシュを消したい。Colab で Drive に置いて再ダウンロードを避けたい。

In [1]:
from huggingface_hub import hf_hub_download, scan_cache_dir, snapshot_download

# 1 ファイルだけダウンロード（戻り値はキャッシュ内のパス）
path = hf_hub_download("intfloat/multilingual-e5-small", "config.json")
print(path)

# リポジトリの一部をまとめてダウンロード
local = snapshot_download("intfloat/multilingual-e5-small", allow_patterns=["*.json", "*.txt"])
print(local)

/root/.cache/huggingface/hub/models--intfloat--multilingual-e5-small/snapshots/614241f622f53c4eeff9890bdc4f31cfecc418b3/config.json
/root/.cache/huggingface/hub/models--intfloat--multilingual-e5-small/snapshots/614241f622f53c4eeff9890bdc4f31cfecc418b3


In [2]:
info = scan_cache_dir()
print(f"キャッシュ全体: {info.size_on_disk / 1e9:.2f} GB")
for repo in sorted(info.repos, key=lambda r: -r.size_on_disk)[:5]:
    print(f"  {repo.size_on_disk / 1e6:8.1f} MB  {repo.repo_type}  {repo.repo_id}")

キャッシュ全体: 0.02 GB
      17.1 MB  model  intfloat/multilingual-e5-small


In [3]:
# 特定のモデルをキャッシュから削除する
repo = next(r for r in info.repos if r.repo_id == "intfloat/multilingual-e5-small")
strategy = info.delete_revisions(*[rev.commit_hash for rev in repo.revisions])
print("解放される容量:", strategy.expected_freed_size_str)
strategy.execute()

解放される容量: 17.1M


- 既定のキャッシュ先は `~/.cache/huggingface/hub`。環境変数 `HF_HOME` で変更できる（import より前に設定する）。
- Colab で Drive に置く例: `os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"`。ただし Drive は読み書きが遅く、容量も消費する。
- 個別に指定するなら `from_pretrained(name, cache_dir=...)`。